In [4]:
# Python 字典基础用法
# 1. 直接创建一个字典，key 为水果类别，value 为相应的数量
fruit_count = {"apple": 5, "banana": 3, "orange": 7}
print("1. 创建一个字典：", fruit_count)

# 2. 取字典中的值
apple_num = fruit_count["apple"] # apple 键不存在就会报错 
apple_num = fruit_count.get("apple", 0) # 安全访问键值：字符不存在返回 0，存在取出已有计数值
banana_num = fruit_count["banana"]
print("2. apple数量：", apple_num, "，banana数量：", banana_num)

# 3. 修改字典里的值
fruit_count["banana"] = 10
print("3. 修改 banana 数量之后的字典：", fruit_count)

# 4. 新增键值对
fruit_count["grape"] = 15
print("4. 新增 grape 键值对之后：", fruit_count)

# 5. 遍历所有的键值对
print("5. 遍历全部水果统计键值对：")
for key, value in fruit_count.items():
    print(f"  水果：{key}，数量：{value}")


1. 创建一个字典： {'apple': 5, 'banana': 3, 'orange': 7}
2. apple数量： 5 ，banana数量： 3
3. 修改 banana 数量之后的字典： {'apple': 5, 'banana': 10, 'orange': 7}
4. 新增 grape 键值对之后： {'apple': 5, 'banana': 10, 'orange': 7, 'grape': 15}
5. 遍历全部水果统计键值对：
  水果：apple，数量：5
  水果：banana，数量：10
  水果：orange，数量：7
  水果：grape，数量：15


In [ ]:
import torch
import math
from collections.abc import Callable, Iterable
from typing import Optional

# 这套 API 约定同样适用于下一节的 AdamW
class SGD(torch.optim.Optimizer):
    '''
    PyTorch 中的优化器统一继承 torch.optim.Optimizer，需要实现两个方法：
    __init__(self, params, ...): 使用基类的 init 方法保存待优化参数和默认超参；
    step(self): 执行一次参数更新
    '''

    def __init__(self, params, lr=1e-3):
        """
        自定义SGD优化器构造函数
        :param params: 待优化的参数集合，可以是模型全部参数，也可以是分组参数
        :param lr: 学习率，超参数
        """
        if lr < 0:
            raise ValueError(f"Invalid learning rate: {lr}")
        defaults = {"lr": lr} # 以字典的形式存储超参名（key）与超参默认值（value)
        super().__init__(params, defaults)

    def step(self, closure: Optional[Callable] = None):
        """
        执行单步参数更新，反向传播完成 grad 计算之后调用此函数
        :param closure: 可选的闭包函数，用于重新计算 loss 并返回 loss 值，部分优化器需要使用（我们不用）
        :return: 如果传入 closure，返回 loss，否则返回 None
        """
        loss = None if closure is None else closure()
        for group in self.param_groups: # 遍历不同的参数组: 不同组可以指定不同的超参数，比如第一层参数用小一点的 lr，最后一层用大一点的 lr
            lr = group["lr"] # group 也是一个字典，取其中的 lr
            for p in group["params"]: # 遍历该组内的每个待优化参数 p
                if p.grad is None: # 梯度还没有计算出来，grad 属性为 None，跳过更新
                    continue

                state = self.state[p]  # self.state 字典：保存每个参数对应的状态信息，如迭代步数、动量等；state[p] 获取参数 p 对应的状态字典
                t = state.get("t", 0)  # 读取迭代计数 t；如果key "t" 不存在，返回默认值 0
                grad = p.grad.data  # 获取损失 loss 关于参数 p 的梯度张量
                p.data -= lr / math.sqrt(t + 1) * grad  # 原地更新参数：学习率随迭代步数做平方根衰减
                state["t"] = t + 1  # 更新状态里的迭代计数，步数 +1

        return loss

In [5]:
# Problem 12: 学习率超参调优 (1 分)
# 用下面这个 toy 例子分别跑 lr=1e1 / 1e2 / 1e3，观察损失的变化：
# 当 lr=1e1，模型可以正常收敛，收敛速度较慢，损失值从 24.18 降低到了 3.25；
# 当 lr=1e2，模型可以正常收敛，收敛速度变快很多，损失值很快降低到 2e-23；
# 当 lr=1e3，模型无法收敛，损失值不断变大，一直到 2e18；

# 测试 SGD：对比不同学习率下 toy 损失的收敛情况
for lr in [1e1, 1e2, 1e3]: # 1*10 1*10^2 1*10^3
    weights = torch.nn.Parameter(5 * torch.randn((10, 10)))
    opt = SGD([weights], lr=lr)

    print(f"--- lr={lr} ---")

    for t in range(10):
        opt.zero_grad()             # 清空上一步的梯度
        loss = (weights**2).mean()  # 构造一个标量损失
        loss.backward()             # 反向传播，计算梯度
        opt.step()                  # 优化器更新参数
        if t % 3 == 0 or t == 9:
            print(f"  step {t:2d}: loss = {loss.item():.3e}")


--- lr=10.0 ---
  step  0: loss = 2.198e+01
  step  3: loss = 8.112e+00
  step  6: loss = 4.595e+00
  step  9: loss = 2.954e+00
--- lr=100.0 ---
  step  0: loss = 2.613e+01
  step  3: loss = 1.073e-01
  step  6: loss = 5.158e-20
  step  9: loss = 2.929e-23
--- lr=1000.0 ---
  step  0: loss = 2.613e+01
  step  3: loss = 1.812e+08
  step  6: loss = 4.756e+13
  step  9: loss = 2.422e+18


In [6]:
# Problem 13:  实现 AdamW (2 分)
class AdamW(torch.optim.Optimizer):
    def __init__(self, params, lr:float = 1e-3, betas:tuple[float, float] = (0.9, 0.999), eps:float = 1e-8, weight_decay:float = 0.0):
        '''
        lr: float  α, learning‑rate 学习率
        betas: tuple[float, float] (β1, β2), 一阶矩、二阶矩估计的指数衰减系数
        eps: float ε, 保证数值稳定性的小常数
        weight_decay: float λ, weight‑decay 权重衰减系数
        '''
        if lr < 0:
            raise ValueError(f"Invalid learning rate: {lr}")

        # defaults 字典保存优化器的默认超参数
        defaults = {"lr": lr, "betas": betas, "eps": eps, "weight_decay": weight_decay}

        # 调用父类 Optimizer 的构造函数
        super().__init__(params, defaults)

    def step(self, closure: Optional[Callable] = None):
        """
        执行单步 AdamW 参数更新，反向传播计算完梯度后调用
        :param closure: 可选闭包函数，用于重计算 loss
        :return: loss，若传入 closure 返回 loss，否则返回 None
        """
        # 如果提供 closure，则执行闭包获取 loss，否则 loss 置为 None
        loss = None if closure is None else closure()

        # 遍历每一组参数 group
        for group in self.param_groups:
            # 获取本组参数使用的各种超参
            lr = group["lr"]  
            beta1, beta2 = group["betas"]
            eps = group["eps"]
            weight_decay = group["weight_decay"]

            # 遍历当前参数组内每一个待优化参数 p
            for p in group["params"]:
                # 如果该参数没有梯度张量，跳过更新
                if p.grad is None:
                    continue
                state = self.state[p]  # 获取参数 p 对应的状态字典，保存 m/v/t 等状态变量
                t = state.get("t", 0)  # 从状态字典读取迭代步数，初值为 0
                m = state.get("m", 0) # 一阶矩向量（first moment vector）
                v = state.get("v", 0) # 二阶矩向量（second moment vector）
                grad = p.grad.data  # 获取损失关于参数 p 的梯度张量

                t += 1 # AdawW 里面 t 的取值范围为 1~T，代码里它的初值是 0，所以需要先加 1，再执行更新

                # 执行 AdamW 单步更新
                p.data -= lr * weight_decay * p.data # AdamW 的权重衰减：单独对权重做衰减

                m_t = beta1 * m + (1 - beta1) * grad # 更新一阶矩估计
                v_t = beta2 * v + (1 - beta2) * grad ** 2 # 更新二阶矩估计
                m_hat = m_t / (1 - beta1 ** t) # 用 α 修正因子的分母调节一阶矩估计
                v_hat = v_t / (1 - beta2 ** t) # 用 α 修正因子的分子调节二阶矩估计
                p.data -= lr * m_hat / (torch.sqrt(v_hat) + eps) # 执行 moment-adjusted 的权重更新

                state["t"] = t  # 更新状态字典中的迭代步数
                state["m"] = m_t # 保存最新一阶矩到状态
                state["v"] = v_t # 保存最新二阶矩到状态

        return loss


In [8]:
# 测试 AdamW：用同一个 toy 损失观察优化过程
weights = torch.nn.Parameter(5 * torch.randn((10, 10)))
opt = AdamW([weights], lr=1e2, weight_decay=0.01)

for t in range(20):
    opt.zero_grad()
    loss = (weights**2).mean()
    loss.backward()
    opt.step()
    if t % 5 == 0 or t == 19:
        print(f"step {t:2d}: loss = {loss.item():.4f}")


step  0: loss = 31.9408
step  5: loss = 1.1764
step 10: loss = 0.0007
step 15: loss = 0.0000
step 19: loss = 0.0000


In [11]:
# Problem 14:  实现带预热的余弦学习率调度 (1 分)
def learning_rate_schedule(
    it: int,
    max_learning_rate: float,
    min_learning_rate: float,
    warmup_iters: int,
    cosine_cycle_iters: int,
):
    '''
    it: int  当前迭代步数 t (t 从 0 开始); 实际训练时 t 应该是从 1 开始的
    max_learning_rate: float  α_max，最大学习率
    min_learning_rate: float  α_min，最小（最终）学习率
    warmup_iters: int  T_w，预热迭代数
    cosine_cycle_iters: int  T_c，余弦退火的结束步数
    return: float，第 it 步使用的学习率 α_t
    '''

    # 预热阶段：学习率从 0 线性升到 max_learning_rate
    if it < warmup_iters:
        return it / warmup_iters * max_learning_rate
    # 余弦退火阶段：按余弦曲线从 max_learning_rate 衰减到 min_learning_rate
    elif warmup_iters <= it <= cosine_cycle_iters:
        return min_learning_rate + 0.5 * (1 + math.cos((it - warmup_iters) / (cosine_cycle_iters - warmup_iters) * math.pi)) * (max_learning_rate - min_learning_rate)
    # 退火结束：保持 min_learning_rate
    else:
        return min_learning_rate


In [12]:
# 测试 learning_rate_schedule：打印几个关键节点的学习率
max_lr, min_lr, warmup, cycle = 1e-3, 1e-4, 100, 1000
for it in [0, 50, 100, 500, 1000, 1200]:
    lr_t = learning_rate_schedule(it, max_lr, min_lr, warmup, cycle)
    print(f"it={it:5d}   lr={lr_t:.6f}")


it=    0   lr=0.000000
it=   50   lr=0.000500
it=  100   lr=0.001000
it=  500   lr=0.000628
it= 1000   lr=0.000100
it= 1200   lr=0.000100


In [15]:
# Problem 15:  实现梯度裁剪 (1 分)
def gradient_clipping(parameters: Iterable[torch.nn.Parameter], max_l2_norm: float, eps:float = 1e-6) -> None:

    # 把所有参数的梯度当成一个大向量，计算整体的 ℓ2 范数
    # 1. p.grad.pow(2).sum().item(): 对单个参数 梯度张量内的所有元素求平方和，得到该参数张量 梯度的平方和
    # 2. sum(... for p in parameters): 对所有参数张量的梯度平方和求和，得到所有参数的梯度的平方和
    l2_norm = math.sqrt(sum(p.grad.pow(2).sum().item() for p in parameters if p.grad is not None))

    # 只在范数超过阈值时整体按比例缩放，方向保持不变（原地修改）
    if l2_norm > max_l2_norm:
        for p in parameters:
            with torch.no_grad():
                if p.grad is not None:
                    p.grad *= max_l2_norm / (l2_norm + eps)


In [16]:
# 测试 gradient_clipping：构造一组偏大的梯度，裁剪后检查整体范数
weights = torch.nn.Parameter(torch.randn(10, 10))
bias = torch.nn.Parameter(torch.randn(10))
loss = (weights ** 2).sum() * 100 + (bias ** 2).sum() * 100
loss.backward()

params = [weights, bias]
before = math.sqrt(sum(p.grad.pow(2).sum().item() for p in params))
gradient_clipping(params, max_l2_norm=1.0)
after = math.sqrt(sum(p.grad.pow(2).sum().item() for p in params))

print(f"裁剪前梯度范数：{before:.4f}")
print(f"裁剪后梯度范数：{after:.10f}（应略小于 1.0）")


裁剪前梯度范数：1976.0170
裁剪后梯度范数：1.0000000373（应略小于 1.0）
